# NB-08: CausalConv3d, ResidualBlock, and AttentionBlock

## Learning Objectives
- Understand CausalConv3d's asymmetric temporal padding and why it ensures causality
- Trace ResidualBlock's skip-connection with automatic dimension projection
- Compare the VAE's single-head spatial AttentionBlock with the DiT's multi-head attention

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm concept -- VAE uses its own channel-first variant)
- **Assumed concepts:** nn.Conv3d, F.pad, skip connections, self-attention

## Concept Map
- **CausalConv3d** -> used in every temporal convolution in Encoder3d (NB-09) and Decoder3d (NB-10)
- **ResidualBlock** -> stacked 2x per encoder level, 3x per decoder level
- **AttentionBlock** -> used once in encoder middle block and once in decoder middle block

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for VAE demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_vae.py directly
_vae_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_vae.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_vae", _vae_path)
vae_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_vae"] = vae_mod
_spec.loader.exec_module(vae_mod)

from diffsynth.models.wan_video_vae import (
    CausalConv3d, RMS_norm, ResidualBlock, AttentionBlock, Resample
)
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Setup complete.")

## 1. CausalConv3d: Left-Only Temporal Padding

Standard Conv3d with `kernel_size=(3,3,3)` pads symmetrically in all dimensions. Each output frame depends on past **and** future frames -- this is non-causal and unsuitable for autoregressive video generation.

CausalConv3d fixes this by remapping the temporal padding so it is entirely on the "past" side (left-only). The spatial dimensions (H, W) keep their symmetric padding. This ensures each output frame depends only on current and past frames -- critical for sequential generation.

The key insight is the mapping between Conv3d's `padding=(t, h, w)` parameter and the F.pad tuple format. Conv3d stores padding in `(t, h, w)` order, but `F.pad` expects the **reversed** dimension order: `(w_left, w_right, h_left, h_right, t_left, t_right)`. CausalConv3d exploits this by setting `t_left = 2*t` and `t_right = 0`, making the temporal padding asymmetric (causal) while keeping spatial padding symmetric.

### Temporal Padding: Standard vs Causal

```
Standard (symmetric) Conv3d padding with kernel_t=3:
                     pad=1         pad=1
Frames: ... | [0] | [F0] | [F1] | [F2] | [F3] | [F4] | [0] | ...
                     ^                     ^         
              Output F1 sees:       Output F3 sees:
              F0, F1, F2            F2, F3, F4
              (past + future!)      (past + future!)

Causal (left-only) Conv3d padding with kernel_t=3:
              pad=2                                    pad=0
Frames: [0] | [0] | [F0] | [F1] | [F2] | [F3] | [F4]
                     ^                     ^         
              Output F0 sees:       Output F2 sees:
              0, 0, F0              F0, F1, F2
              (only past + current) (only past + current)
```

### Padding Tuple Mapping

```
Conv3d padding=(1, 1, 1)  [t, h, w order]
           |
           v  CausalConv3d.__init__ remaps to F.pad format:
F.pad _padding=(1, 1, 1, 1, 2, 0)  [w_l, w_r, h_l, h_r, t_l, t_r]
                                     ^^spatial symmetric^^  ^^causal^^
```

Notice the **reversed dimension order**: Conv3d thinks `(t, h, w)` but F.pad works from the **last** dimension inward: `(w, h, t)`. CausalConv3d doubles the temporal left-padding (`2 * t`) and sets right-padding to 0, achieving causal behavior.

In [ ]:
# Source: wan_video_vae.py, line 33
# Annotated walkthrough of CausalConv3d:
#
# class CausalConv3d(nn.Conv3d):
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         # Remap padding: spatial stays symmetric, temporal becomes left-only
#         # self.padding from nn.Conv3d is (t, h, w)
#         # F.pad expects (w_l, w_r, h_l, h_r, t_l, t_r) -- reversed!
#         self._padding = (self.padding[2], self.padding[2],   # w: symmetric
#                          self.padding[1], self.padding[1],   # h: symmetric
#                          2 * self.padding[0], 0)             # t: all left, none right
#         self.padding = (0, 0, 0)  # zero out Conv3d's own padding
#
#     def forward(self, x, cache_x=None):
#         # (cache_x handling omitted -- runtime optimization, per D-02)
#         x = F.pad(x, list(self._padding))
#         return super().forward(x)

print("CausalConv3d: extends nn.Conv3d, remaps temporal padding to be causal (left-only)")
print("Key: F.pad uses REVERSED dimension order compared to Conv3d padding parameter")

In [ ]:
# Source: wan_video_vae.py, line 33
# Demonstrate causality: each output frame depends only on current and past frames

# Create CausalConv3d with kernel_size=(3,1,1) -- temporal-only kernel for clarity
conv = CausalConv3d(1, 1, kernel_size=(3, 1, 1), padding=(1, 0, 0))

# Set weights to identity (all ones) for interpretable output
with torch.no_grad():
    conv.weight.fill_(1.0)
    conv.bias.fill_(0.0)

# Input: 5 frames with values 1,2,3,4,5
x = torch.arange(1, 6, dtype=torch.float32).reshape(1, 1, 5, 1, 1)
print(f"Input shape: {x.shape}  (B=1, C=1, T=5, H=1, W=1)")
print(f"Input values: {x.squeeze().tolist()}")

out = conv(x)
assert out.shape == x.shape, f"Temporal dim preserved: {out.shape} == {x.shape}"
print(f"\nOutput shape: {out.shape}  (temporal dimension PRESERVED)")
print(f"Output values: {out.squeeze().tolist()}")
print()
print("Causality verification:")
print(f"  Frame 0: 0 + 0 + 1 = 1   (two zero pads from left)")
print(f"  Frame 1: 0 + 1 + 2 = 3   (one zero pad)")
print(f"  Frame 2: 1 + 2 + 3 = 6   (all real frames -- NO future frames used)")
print(f"  Frame 3: 2 + 3 + 4 = 9")
print(f"  Frame 4: 3 + 4 + 5 = 12")

In [ ]:
# Source: wan_video_vae.py, line 33
# Full CausalConv3d with spatial dims -- reduced config demo

# VAE reduced config
dim = 32
B, C_in, T, H, W = 1, 3, 5, 64, 64

# CausalConv3d as used in encoder conv1: (3 -> dim, kernel=3, padding=1)
conv1 = CausalConv3d(C_in, dim, kernel_size=3, padding=1)

x = torch.randn(B, C_in, T, H, W)
print(f"Input shape:  {x.shape}")
print(f"Conv3d padding (original): {(1, 1, 1)}")
print(f"F.pad _padding (remapped): {conv1._padding}")

with torch.no_grad():
    out = conv1(x)

assert out.shape == torch.Size([B, dim, T, H, W]), f"Expected {(B, dim, T, H, W)}, got {out.shape}"
print(f"Output shape: {out.shape}")
print(f"\nKey observations:")
print(f"  - Temporal dim preserved: T_in={T} == T_out={out.shape[2]}")
print(f"  - Spatial dims preserved: H={H}, W={W} (padding=1 compensates kernel=3)")
print(f"  - Channels changed: {C_in} -> {dim}")
print(f"  - Parameters: {sum(p.numel() for p in conv1.parameters()):,}")

## 2. RMS_norm: Channel-First Normalization

The VAE uses its own `RMS_norm` variant that differs from the DiT's `RMSNorm` (NB-01). Both compute the same core operation -- `x * rsqrt(mean(x^2) + eps) * gamma` -- but they operate on different tensor layouts:

- **VAE `RMS_norm(dim, channel_first=True, images=False)`**: normalizes along `dim=1` (channel axis) for 5D `[B, C, T, H, W]` tensors. The learnable `gamma` has shape `(dim, 1, 1, 1)` so it broadcasts over the T, H, W dimensions.
- **DiT `RMSNorm(dim)`**: normalizes along `dim=-1` (feature axis) for 3D `[B, S, D]` tensors. The learnable `weight` has shape `(dim,)`.

The difference stems from how the two architectures lay out their tensors: the DiT works in sequence format `[B, S, D]` (channel-last), while the VAE works in image/video format `[B, C, T, H, W]` (channel-first).

Source: `wan_video_vae.py`, line 55

In [ ]:
# Source: wan_video_vae.py, line 55
# VAE's channel-first RMS_norm for 5D tensors

norm_vae = RMS_norm(dim=32, channel_first=True, images=False)
x = torch.randn(1, 32, 5, 8, 8)  # [B, C, T, H, W]
out = norm_vae(x)
assert out.shape == x.shape
print(f"VAE RMS_norm: input {x.shape} -> output {out.shape}")
print(f"  gamma shape: {norm_vae.gamma.shape}  (broadcasts over T, H, W)")
print(f"  Normalizes along: dim=1 (channel axis)")
print(f"  Parameters: {sum(p.numel() for p in norm_vae.parameters()):,}")

## 3. ResidualBlock: Skip Connection with Dimension Projection

The ResidualBlock is the workhorse of the VAE encoder and decoder. It applies two CausalConv3d layers with RMS_norm + SiLU activations, wrapped in a residual (skip) connection:

```
forward(x) = shortcut(x) + body(x)
```

**The body path:** RMS_norm -> SiLU -> CausalConv3d(in_dim, out_dim, 3) -> RMS_norm -> SiLU -> Dropout -> CausalConv3d(out_dim, out_dim, 3)

**The shortcut path:**
- When `in_dim != out_dim`: a 1x1 CausalConv3d projection `CausalConv3d(in_dim, out_dim, 1)` matches the channel dimensions
- When `in_dim == out_dim`: the shortcut is `nn.Identity()` -- zero extra parameters

This pattern is used 2x per encoder level and 3x per decoder level throughout the VAE.

Source: `wan_video_vae.py`, lines 267-301

In [ ]:
# Source: wan_video_vae.py, lines 267-301
# Case 1: Same dimensions (no shortcut projection)
res_same = ResidualBlock(in_dim=32, out_dim=32)
x = torch.randn(1, 32, 5, 32, 32)
with torch.no_grad():
    out_same = res_same(x)
assert out_same.shape == x.shape
print(f"ResidualBlock(32 -> 32): {x.shape} -> {out_same.shape}")
print(f"  Shortcut: Identity (in_dim == out_dim)")
print(f"  Parameters: {sum(p.numel() for p in res_same.parameters()):,}")

# Case 2: Different dimensions (1x1 CausalConv3d shortcut projection)
res_diff = ResidualBlock(in_dim=32, out_dim=64)
with torch.no_grad():
    out_diff = res_diff(x)
assert out_diff.shape == torch.Size([1, 64, 5, 32, 32])
print(f"\nResidualBlock(32 -> 64): {x.shape} -> {out_diff.shape}")
print(f"  Shortcut: CausalConv3d(32, 64, kernel_size=1)")
print(f"  Parameters: {sum(p.numel() for p in res_diff.parameters()):,}")
print(f"  Extra params from shortcut: {sum(p.numel() for p in res_diff.shortcut.parameters()):,}")

## 4. AttentionBlock: Per-Frame Spatial Self-Attention

The VAE's AttentionBlock applies single-head self-attention to each video frame independently. Unlike the DiT's multi-head attention that operates over the full spatiotemporal sequence, this block:

1. **Operates per-frame**: reshapes `[B, C, T, H, W]` -> `[B*T, C, H, W]` so each frame is processed independently
2. **Single head**: Q, K, V all have shape `[B*T, 1, H*W, C]` -- only one attention head
3. **Spatial tokens**: each spatial position (pixel) is a token, so attention learns global spatial relationships within each frame
4. **Zero-initialized output projection**: `nn.init.zeros_(self.proj.weight)` -- at initialization, the learned attention features contribute nothing through the projection (weights are all zero), so the block starts as near-identity via the residual connection
5. **Uses `F.scaled_dot_product_attention`** for numerical stability

This block is used only once in the encoder middle block and once in the decoder middle block -- it provides global spatial context at the bottleneck resolution.

Source: `wan_video_vae.py`, lines 304-342

In [ ]:
# Source: wan_video_vae.py, lines 304-342
# Annotated walkthrough of AttentionBlock.forward:
#
# 1. Save residual: h = x
# 2. Reshape 5D -> 4D: [B,C,T,H,W] -> [B*T, C, H, W]  (per-frame processing)
# 3. RMS_norm (channel-first)
# 4. to_qkv: Conv2d(dim, dim*3, 1) -> chunk into Q, K, V  each [B*T, C, H, W]
# 5. Reshape to attention format: [B*T, 1, H*W, C]  (1 head, H*W spatial tokens)
# 6. F.scaled_dot_product_attention(Q, K, V) -> [B*T, 1, H*W, C]
# 7. Reshape back: [B*T, C, H, W]
# 8. proj: Conv2d(dim, dim, 1) with zero-init weights
# 9. Reshape 4D -> 5D: [B*T, C, H, W] -> [B, C, T, H, W]
# 10. Residual: return h + x

attn = AttentionBlock(dim=32)
x = torch.randn(1, 32, 5, 8, 8)
print(f"Input shape: {x.shape}  [B, C, T, H, W]")

with torch.no_grad():
    out = attn(x)

assert out.shape == x.shape, f"Shape preserved: {out.shape} == {x.shape}"
print(f"Output shape: {out.shape}  (shape PRESERVED -- residual connection)")
print(f"\nInternal reshape: [1,32,5,8,8] -> [5, 32, 8, 8] (B*T frames)")
print(f"Attention tokens: H*W = 8*8 = 64 spatial positions per frame")
print(f"Attention heads: 1 (single head)")
print(f"Parameters: {sum(p.numel() for p in attn.parameters()):,}")
print(f"  to_qkv (Conv2d 32->96, 1x1): {sum(p.numel() for p in attn.to_qkv.parameters()):,}")
print(f"  proj (Conv2d 32->32, 1x1, zero-init): {sum(p.numel() for p in attn.proj.parameters()):,}")

### VAE AttentionBlock vs DiT SelfAttention

| Property | VAE AttentionBlock | DiT SelfAttention (NB-04) |
|----------|-------------------|---------------------------|
| Heads | 1 (single) | num_heads (4 in reduced config) |
| Operates on | Per-frame spatial: [B*T, 1, H*W, C] | Full sequence: [B, num_heads, S, head_dim] |
| Token meaning | Each spatial position in one frame | Each spatiotemporal patch |
| QKV source | Conv2d(dim, dim*3, 1) | Linear(dim, dim*3) |
| RoPE | No | Yes (3D frequency bands) |
| Output init | proj weight = zeros (identity at start) | Standard initialization |
| Where used | Encoder/decoder middle block only (1x each) | Every DiTBlock (30x in production) |
| Purpose | Global spatial context within each frame | Main processing across all tokens |

In [ ]:
# Source: wan_video_vae.py, line 304
# Verify zero-initialization of proj.weight
attn_fresh = AttentionBlock(dim=32)

# The source zeros only proj.weight (line 319): nn.init.zeros_(self.proj.weight)
# proj.bias remains at default Conv2d init (small values)
assert (attn_fresh.proj.weight == 0).all(), "proj.weight should be all zeros at init"
print(f"proj.weight all zeros: {(attn_fresh.proj.weight == 0).all().item()}")
print(f"proj.bias range: [{attn_fresh.proj.bias.min():.4f}, {attn_fresh.proj.bias.max():.4f}]")

# With zero weights, proj(x) = bias only (input-independent constant)
# The residual becomes: identity + broadcast(bias) -- a near-identity transform
x_test = torch.randn(1, 32, 5, 8, 8)
with torch.no_grad():
    out_fresh = attn_fresh(x_test)

# The difference from input should be the proj.bias broadcast over spatial dims
diff = out_fresh - x_test  # [B, C, T, H, W]
# Each channel's diff should be constant across all spatial positions (just bias)
per_channel_std = diff[0, :, 0, :, :].reshape(32, -1).std(dim=1)
assert per_channel_std.max() < 1e-5, f"Diff should be spatially constant (bias only), got std={per_channel_std.max()}"
print(f"\nPer-channel spatial std of (output - input): max={per_channel_std.max():.2e}")
print("  -> Spatially constant diff confirms output = identity + bias broadcast")
print("\nConfirmed: zero-init proj.weight means learned attention features don't contribute at init")
print("The block starts as near-identity -- training gradually learns spatial attention")

## 5. Resample: Spatial Down/Up Sampling

The Resample module handles spatial resolution changes between encoder/decoder levels. It will be exercised in detail in NB-09 (Encoder) and NB-10 (Decoder), but we introduce its design here:

- **`downsample2d`**: `ZeroPad2d(0, 1, 0, 1)` + `Conv2d(dim, dim, 3, stride=2)` -- halves spatial dimensions, keeps channels unchanged
- **`upsample2d`**: `Upsample(2x, nearest-exact)` + `Conv2d(dim, dim//2, 3, padding=1)` -- doubles spatial dimensions, **halves** channels

**Asymmetric design**: downsample preserves channels, but upsample halves channels. This means the decoder must account for channel halving at each upsample level.

**Why asymmetric padding in downsample**: For even-dimension inputs, the right+bottom `ZeroPad2d(0,1,0,1)` ensures that stride-2 convolution gives exact halving. Without it, a 64x64 input with kernel=3, stride=2 would give 32x32 only with `padding=1`, but the padding is already set to 0 in the Conv2d -- so the ZeroPad2d acts as explicit asymmetric padding: `64 + 1 (right) = 65 -> (65-3)/2 + 1 = 32`.

Source: `wan_video_vae.py`, line 82

## Summary

### Key Takeaways
- **CausalConv3d** remaps `padding=(t, h, w)` to `F.pad=(w_l, w_r, h_l, h_r, 2*t, 0)`, ensuring temporal causality. Each output frame depends only on current and past frames.
- **RMS_norm** (VAE variant) normalizes along dim=1 (channel axis) for `[B, C, T, H, W]` tensors, unlike the DiT's RMSNorm which normalizes along dim=-1 for `[B, S, D]`.
- **ResidualBlock** uses `shortcut(x) + body(x)` with automatic 1x1 CausalConv3d projection when in_dim != out_dim. Used 2x per encoder level, 3x per decoder level.
- **AttentionBlock** operates per-frame with a single head on H*W spatial tokens, zero-initialized to start as near-identity. Used only at the encoder/decoder bottleneck.
- **Resample** downsample2d preserves channels with stride-2 conv; upsample2d doubles spatial dims but halves channels.

### Source References

| Symbol | Location |
|--------|----------|
| CausalConv3d | `wan_video_vae.py`, line 33 |
| RMS_norm | `wan_video_vae.py`, line 55 |
| Resample | `wan_video_vae.py`, line 82 |
| ResidualBlock | `wan_video_vae.py`, line 267 |
| AttentionBlock | `wan_video_vae.py`, line 304 |

## Exercises

### Exercise 1 -- Kernel size exploration
Change the CausalConv3d kernel from `(3, 1, 1)` to `(5, 1, 1)`. What is the new `_padding` tuple? How many past frames does each output frame depend on? (Hint: with kernel_t=5 and padding=(2,0,0), the causal padding becomes `t_left = 2*2 = 4`.)

### Exercise 2 -- ResidualBlock parameter comparison
Create a `ResidualBlock(32, 32)` and a `ResidualBlock(32, 64)`. Compare parameter counts. What fraction of the larger block's parameters are in the shortcut projection? (Hint: the shortcut is `CausalConv3d(32, 64, kernel_size=1)` -- how many parameters does a 1x1x1 3D convolution have?)

### Exercise 3 -- Joint vs per-frame attention memory
The AttentionBlock processes each frame independently. What would happen to memory usage if it instead processed all T frames jointly (i.e., tokens = T*H*W instead of H*W)? Calculate the attention matrix size for T=5, H=64, W=64:
- Per-frame: `64*64 = 4,096` tokens -> attention matrix is `4,096 x 4,096 = 16.8M` entries
- Joint: `5*64*64 = 20,480` tokens -> attention matrix is `20,480 x 20,480 = 419.4M` entries
- That's a 25x increase in memory! Per-frame processing is critical for keeping VAE attention tractable.